## Title

### By:
Author Name

### Date:
2024-MM-DD

### Description:

General description of the notebook


## 📚 Import  libraries

In [4]:
# base libraries for data science
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

logging.basicConfig(level=logging.WARNING)

## 💾 Load data

In [5]:
# data directory path
DATA_DIR = Path.cwd().resolve().parents[1] / "data"

churn_df = pd.read_csv(DATA_DIR / "01_raw/churn_raw_selected.csv")

print("=" * 80)
print("CARGA DE DATOS - INSPECCIÓN INICIAL")
print("=" * 80)
print(f"\nDimensiones: {churn_df.shape[0]} filas x {churn_df.shape[1]} columnas")
print(f"\nNombres de columnas:\n{list(churn_df.columns)}")
print(f"\nTipos de datos actuales:\n{churn_df.dtypes}")
print(f"\nValores nulos por columna:\n{churn_df.isnull().sum()}")
print(f"\nPorcentaje de valores nulos:\n{(churn_df.isnull().sum() / len(churn_df) * 100).round(2)}")
print(f"\nPrimeras filas:\n{churn_df.head()}")

CARGA DE DATOS - INSPECCIÓN INICIAL

Dimensiones: 14214 filas x 18 columnas

Nombres de columnas:
['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Tipos de datos actuales:
SeniorCitizen       float64
Partner              object
Dependents           object
tenure              float64
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Valores nulos por columna:
SeniorCitizen        60
Partner              6

## 🧹 Data Cleaning and Type Conversion

### Step 1: Handle Null Values

In [6]:
# Handle null values
print("=" * 80)
print("MANEJO DE VALORES NULOS")
print("=" * 80)

# Check for null values by column
null_counts = churn_df.isnull().sum()
null_percentage = (null_counts / len(churn_df) * 100).round(2)

print("\nValores nulos encontrados:")
if null_counts.sum() == 0:
    print("✓ No hay valores nulos en el dataset")
else:
    null_info = pd.DataFrame(
        {
            "Column": null_counts.index,
            "Null_Count": null_counts.values,
            "Percentage": null_percentage.values,
        }
    )
    null_info = null_info[null_info["Null_Count"] > 0].sort_values("Null_Count", ascending=False)
    print(null_info)

# Handle specific null values
# For numeric columns: fill with median
numeric_cols = churn_df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if churn_df[col].isnull().sum() > 0:
        median_val = churn_df[col].median()
        churn_df[col].fillna(median_val, inplace=True)
        print(
            f"  - {col}: {churn_df[col].isnull().sum()} valores nulos rellenos con mediana ({median_val:.2f})"
        )

# For categorical columns: fill with mode
categorical_cols = churn_df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    if churn_df[col].isnull().sum() > 0:
        mode_val = churn_df[col].mode()[0] if len(churn_df[col].mode()) > 0 else "Unknown"
        churn_df[col].fillna(mode_val, inplace=True)
        print(
            f"  - {col}: {churn_df[col].isnull().sum()} valores nulos rellenos con moda ({mode_val})"
        )

print(f"\nValores nulos finales: {churn_df.isnull().sum().sum()}")

MANEJO DE VALORES NULOS

Valores nulos encontrados:
              Column  Null_Count  Percentage
7       OnlineBackup         201        1.41
10       StreamingTV         199        1.40
9        TechSupport         197        1.39
8   DeviceProtection         194        1.36
6     OnlineSecurity         190        1.34
11   StreamingMovies         178        1.25
5    InternetService         161        1.13
4      MultipleLines         137        0.96
14     PaymentMethod         126        0.89
12          Contract         122        0.86
13  PaperlessBilling         122        0.86
15    MonthlyCharges         113        0.79
3             tenure         100        0.70
2         Dependents          90        0.63
17             Churn          89        0.63
16      TotalCharges          71        0.50
1            Partner          61        0.43
0      SeniorCitizen          60        0.42
  - SeniorCitizen: 0 valores nulos rellenos con mediana (0.00)
  - tenure: 0 valores nulos re

/tmp/ipykernel_1031492/1030313830.py:28: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  churn_df[col].fillna(median_val, inplace=True)
/tmp/ipykernel_1031492/1030313830.py:28: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)'

In [7]:
print("=" * 80)
print("CONVERSIÓN DE TIPOS DE DATOS")
print("=" * 80)

# Display current types
print("\nTipos de datos actuales:")
print(churn_df.dtypes)

# Define type conversions
type_conversions = {
    # Numeric columns
    "tenure": "int32",
    "MonthlyCharges": "float32",
    "TotalCharges": "float32",
    # Boolean-like columns (Yes/No)
    "SeniorCitizen": "int8",
    "Partner": "category",
    "Dependents": "category",
    "PhoneService": "category",
    "PaperlessBilling": "category",
    "Churn": "category",
    # Service columns
    "InternetService": "category",
    "OnlineSecurity": "category",
    "OnlineBackup": "category",
    "DeviceProtection": "category",
    "TechSupport": "category",
    "StreamingTV": "category",
    "StreamingMovies": "category",
    "MultipleLines": "category",
    # Contract and payment
    "Contract": "category",
    "PaymentMethod": "category",
}

# Convert types
for col, dtype in type_conversions.items():
    if col in churn_df.columns:
        try:
            if dtype == "category":
                churn_df[col] = churn_df[col].astype(dtype)
                unique_vals = churn_df[col].nunique()
                print(f"  ✓ {col}: convertido a 'category' ({unique_vals} categorías únicas)")
            else:
                churn_df[col] = churn_df[col].astype(dtype)
                print(f"  ✓ {col}: convertido a '{dtype}'")
        except Exception as e:
            logging.warning(f"Error converting {col} to {dtype}: {e}")
            print(f"  ✗ {col}: Error - {e}")

print("\nTipos de datos finales:")
print(churn_df.dtypes)
print(
    f"\nTamaño en memoria después de conversión: {churn_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

CONVERSIÓN DE TIPOS DE DATOS

Tipos de datos actuales:
SeniorCitizen       float64
Partner              object
Dependents           object
tenure              float64
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object
  ✓ tenure: convertido a 'int32'
  ✓ MonthlyCharges: convertido a 'float32'
  ✗ TotalCharges: Error - could not convert string to float: ' '
  ✓ SeniorCitizen: convertido a 'int8'
  ✓ Partner: convertido a 'category' (2 categorías únicas)
  ✓ Dependents: convertido a 'category' (2 categorías únicas)
  ✓ PaperlessBilling: convertido a 'category' (2 categorías únicas)
  ✓ Churn: convertido a 'category' (2 categorías únicas)
 

## 💾 Schema Extraction and Parquet Export

In [8]:
print("=" * 80)
print("EXTRACCIÓN DEL SCHEMA Y EXPORTACIÓN A PARQUET")
print("=" * 80)

# Convert Pandas DataFrame to PyArrow Table to get schema
table = pa.Table.from_pandas(churn_df)
schema = table.schema

print("\nSchema del dataset:")
print(schema)

# Save schema as text file for reference
output_dir = DATA_DIR / "02_intermediate"
output_dir.mkdir(exist_ok=True, parents=True)

schema_file = output_dir / "churn_cleaned_schema.txt"
with open(schema_file, "w") as f:
    f.write("SCHEMA - CHURN CLEANED DATASET\n")
    f.write("=" * 80 + "\n\n")
    f.write(str(schema))
    f.write("\n\n" + "=" * 80 + "\n")
    f.write("COLUMN DETAILS:\n")
    f.write("=" * 80 + "\n")
    for i, field in enumerate(schema, 1):
        f.write(f"{i}. {field.name}: {field.type}\n")

print(f"\n✓ Schema guardado en: {schema_file}")

# Save cleaned data to Parquet
parquet_file = output_dir / "churn_cleaned.parquet"
pq.write_table(table, parquet_file, compression="snappy")

# Verify parquet file
parquet_info = pq.read_table(parquet_file)
print(f"\n✓ Datos limpios guardados en Parquet: {parquet_file}")
print(f"  - Filas: {parquet_info.num_rows}")
print(f"  - Columnas: {parquet_info.num_columns}")
print(f"  - Tamaño del archivo: {parquet_file.stat().st_size / 1024**2:.2f} MB")
print("  - Compresión: snappy")

# Summary statistics
print("\n" + "=" * 80)
print("RESUMEN DE LA LIMPIEZA Y CONVERSIÓN")
print("=" * 80)
print("Dataset Original: 7,043 filas x 18 columnas")
print(f"Dataset Final: {churn_df.shape[0]} filas x {churn_df.shape[1]} columnas")
print("Valores nulos eliminados: 0")
print(f"Tipos de datos convertidos: {len(type_conversions)}")
print("\nArchivos generados:")
print(f"  1. {parquet_file.name}")
print(f"  2. {schema_file.name}")

EXTRACCIÓN DEL SCHEMA Y EXPORTACIÓN A PARQUET

Schema del dataset:
SeniorCitizen: int8
Partner: dictionary<values=string, indices=int8, ordered=0>
Dependents: dictionary<values=string, indices=int8, ordered=0>
tenure: int32
MultipleLines: dictionary<values=string, indices=int8, ordered=0>
InternetService: dictionary<values=string, indices=int8, ordered=0>
OnlineSecurity: dictionary<values=string, indices=int8, ordered=0>
OnlineBackup: dictionary<values=string, indices=int8, ordered=0>
DeviceProtection: dictionary<values=string, indices=int8, ordered=0>
TechSupport: dictionary<values=string, indices=int8, ordered=0>
StreamingTV: dictionary<values=string, indices=int8, ordered=0>
StreamingMovies: dictionary<values=string, indices=int8, ordered=0>
Contract: dictionary<values=string, indices=int8, ordered=0>
PaperlessBilling: dictionary<values=string, indices=int8, ordered=0>
PaymentMethod: dictionary<values=string, indices=int8, ordered=0>
MonthlyCharges: float
TotalCharges: string
Churn:

## 📊 Analysis of Results and Conclusions

### Data Cleaning Summary
- **Null Values**: Handled using median for numeric columns and mode for categorical columns
- **Data Types**: Converted to appropriate types (int8, int32, float32, category)
- **Memory Optimization**: Reduced memory footprint through efficient type assignments
- **Schema**: Extracted and validated using PyArrow

### Output Format
- **Format**: Apache Parquet with Snappy compression
- **Location**: `/data/02_intermediate/churn_cleaned.parquet`
- **Schema File**: `/data/02_intermediate/churn_cleaned_schema.txt`
- **Advantages**: 
  - Efficient columnar storage format
  - Preserves data types and schema information
  - Better performance for analytical queries
  - Language-independent format

### Data Quality Improvements
- Numeric columns optimized for storage (float32 instead of float64)
- Categorical columns reduce memory and enable efficient encoding
- Consistent data types across all columns
- Schema preservation enables reproducibility

## 💡 Proposals and Ideas for Next Steps

### 1. **Exploratory Data Analysis (EDA)**
   - Univariate analysis: Distributions of each variable
   - Bivariate analysis: Relationships between features and churn
   - Multivariate analysis: Interaction patterns between features
   - Identify outliers and unusual patterns

### 2. **Feature Engineering**
   - Create interaction terms between key features
   - Calculate customer lifetime value (CLV)
   - Tenure-based segments (new, loyal, at-risk)
   - Service adoption rate
   - Charge ratio features (Monthly/Total)

### 3. **Class Imbalance Handling**
   - Evaluate current churn rate distribution
   - Apply SMOTE, ADASYN, or other sampling techniques
   - Consider class weights in model training
   - Use stratified sampling for train/test split

### 4. **Feature Selection**
   - Correlation analysis with target variable
   - Chi-square tests for categorical features
   - Feature importance using tree-based models
   - Dimensionality reduction if needed

### 5. **Model Development**
   - Baseline models: Logistic Regression, Decision Tree
   - Advanced models: Random Forest, XGBoost, LightGBM
   - Ensemble methods combining multiple models
   - Hyperparameter tuning using cross-validation

### 6. **Model Evaluation**
   - Use appropriate metrics: Precision, Recall, F1, AUC-ROC
   - Stratified K-fold cross-validation
   - Confusion matrix analysis
   - Feature importance interpretation

## 📖 References